In [2]:
pip install numpy pandas matplotlib seaborn scikit-learn torch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random
import torch

random.seed(42)
torch.manual_seed(42)
warnings.filterwarnings('ignore')

In [ ]:
""" EDA """

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# --- Minimal Cleaning for EDA ---
# Remove invalid ages
train_df = train_df[train_df['founder_age'] >= 18].reset_index(drop=True)

# Encode Target for Analysis (0=Left, 1=Stayed)
le = LabelEncoder()
train_df['target_encoded'] = le.fit_transform(train_df['retention_status']) # 0: Left, 1: Stayed

# Fill NaNs with median/mode just for visualization purposes
num_cols = train_df.select_dtypes(include=[np.number]).columns
train_df[num_cols] = train_df[num_cols].fillna(train_df[num_cols].median())

print("TRAINING DATA STATISTICS")
print(train_df.describe())
print("\n")
print(train_df.info())

print("TEST DATA STATISTICS")
print(test_df.describe())
print("\n")
print(test_df.info())


print("\nGenerating Correlation Matrix...")

# Select only numerical columns for correlation
numeric_df = train_df.select_dtypes(include=[np.number])

# Compute Correlation
corr_matrix = numeric_df.corr()

# Plot Heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap='RdYlGn',
    linewidths=0.5,
    square=True
)
plt.title('Correlation Matrix of Features', fontsize=16)
plt.tight_layout()
plt.savefig('heatmap.png')
plt.show()


print("Generating t-SNE Plot...")

# Prepare Data
# Drop target, ID, and encoded target to strictly use features
drop_cols = ['retention_status', 'target_encoded', 'founder_id']
X_eda = train_df.drop(columns=drop_cols, axis=1, errors='ignore')
y_eda = train_df['target_encoded']

# Standardize
scaler = StandardScaler()
# Select only numeric features for t-SNE to avoid one-hot dimensionality explosion
X_eda_numeric = X_eda.select_dtypes(include=[np.number])
X_scaled = scaler.fit_transform(X_eda_numeric)

# Compute t-SNE (Using a sample if dataset > 10k to save time, otherwise full)
# For full dataset this might take a minute.
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
tsne_results = tsne.fit_transform(X_scaled)

# Plot t-SNE
custom_colors = {
    0: '#D62728',  # Left (Red)
    1: '#2CA02C'   # Stayed (Green)
}

plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=tsne_results[:, 0],
    y=tsne_results[:, 1],
    hue=y_eda,
    palette=custom_colors,
    s=70,
    alpha=0.6,
    edgecolor='k',
    linewidth=0.3
)

plt.title('t-SNE Projection: Retention Status', fontsize=16)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend(title='Status (0=Left, 1=Stayed)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('tsne_plot.png')
plt.show()

print("Generating Box Plots...")
# 0 = Left, 1 = Stayed
cluster_order = [0, 1]

# Box Plot for Monthly Revenue
plt.figure(figsize=(10, 6))
sns.boxplot(
    x='target_encoded',
    y='monthly_revenue_generated',
    data=train_df,
    order=cluster_order,
    hue='target_encoded',
    palette=custom_colors,
    dodge=False
)
plt.legend([],[], frameon=False)
plt.title('Distribution of Revenue by Retention Status', fontsize=14)
plt.xticks([0, 1], ['Left', 'Stayed'])
plt.xlabel('Retention Status')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.savefig('boxplot_revenue.png')
plt.show()

# Box Plot for Tenure (Years with Startup)
plt.figure(figsize=(10, 6))
sns.boxplot(
    x='target_encoded',
    y='years_with_startup',
    data=train_df,
    order=cluster_order,
    hue='target_encoded',
    palette=custom_colors,
    dodge=False
)
plt.legend([],[], frameon=False)
plt.title('Distribution of Tenure by Retention Status', fontsize=14)
plt.xticks([0, 1], ['Left', 'Stayed'])
plt.xlabel('Retention Status')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.savefig('boxplot_tenure.png')
plt.show()

print("EDA Completed Successfully.")


In [ ]:
""" DATA PREPROCESSING """

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler,RobustScaler,PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Removing Illogical Data
train_df = train_df[train_df['founder_age'] >= 18].reset_index(drop=True)

train_df = train_df.drop(columns='founder_id',axis=1)
test_ids = test_df['founder_id']
test_df = test_df.drop(columns='founder_id',axis=1)

# Removing NaN values

numeric_null_columns = ['monthly_revenue_generated','num_dependents','years_since_founding']
cat_null_columns = ['work_life_balance_rating','venture_satisfaction','team_size_category','education_background']

numeric_imputer = SimpleImputer(strategy='median')
train_df[numeric_null_columns] = numeric_imputer.fit_transform(train_df[numeric_null_columns])
test_df[numeric_null_columns] = numeric_imputer.transform(test_df[numeric_null_columns])

cat_imputer = SimpleImputer(strategy='most_frequent')
train_df[cat_null_columns] = cat_imputer.fit_transform(train_df[cat_null_columns])
test_df[cat_null_columns] = cat_imputer.transform(test_df[cat_null_columns])


X = train_df.drop(columns='retention_status',axis=1)
y = train_df['retention_status']
X_test = test_df

# Dropping unecessary columns

cols_to_drop = ['founder_role','founder_visibility']
X = X.drop(columns=cols_to_drop,axis=1)
X_test = X_test.drop(columns=cols_to_drop,axis=1)

# Encoding the non-numeric data

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

ordinal_cat_cols = [
    'education_background',
    'work_life_balance_rating',
    'startup_performance_rating',
    'startup_stage',
    'team_size_category',
    'startup_reputation',
]

binary_cat_cols = [
    'founder_gender',
    'working_overtime',
    'remote_operations',
    'leadership_scope',
    'innovation_support'
]

nominal_cat_cols = [
    'personal_status',
    'venture_satisfaction'
]


# Ordinal mappings
for df in [X,X_test]:
    df['education_background'] = df['education_background'].map({'High School': 1, 'Associate Degree': 2, "Bachelor’s Degree": 3, "Master’s Degree": 4, 'PhD': 5})
    df['startup_performance_rating'] = df['startup_performance_rating'].map({'Low': 1, 'Below Average': 2, 'Average': 3, 'High': 4})
    df['work_life_balance_rating'] = df['work_life_balance_rating'].map({'Poor': 1, 'Fair': 2, 'Good': 3, 'Excellent': 4})
    df['venture_satisfaction'] = df['venture_satisfaction'].map({'Low': 1, 'Medium': 2, 'High': 3, 'Very High': 4})
    df['startup_stage'] = df['startup_stage'].map({'Entry': 1, 'Mid': 2, 'Senior': 3})
    df['team_size_category'] = df['team_size_category'].map({'Small': 1, 'Medium': 2, 'Large': 3})
    df['startup_reputation'] = df['startup_reputation'].map({'Poor': 1, 'Fair': 2, 'Good': 3, 'Excellent': 4})
    #df['founder_visibility'] = df['founder_visibility'].map({'Low': 1, 'Medium': 2, 'High': 3, 'Very High': 4})

preprocessor = ColumnTransformer(
    transformers=[
        ('binary', OrdinalEncoder(), binary_cat_cols),
        ('nominal', OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore'), nominal_cat_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform="pandas")

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])
X = pipeline.fit_transform(X)
X_test = pipeline.transform(X_test)

# This ensures they are Integers, while Revenue/Age remain Floats.
X[ordinal_cat_cols+binary_cat_cols] = X[ordinal_cat_cols+binary_cat_cols]
X_test[ordinal_cat_cols+binary_cat_cols] = X_test[ordinal_cat_cols+binary_cat_cols]

# Feature Engineering

X['age_x_experience'] = X['founder_age'] * X['years_with_startup']
X['revenue_per_year'] = X['monthly_revenue_generated'] / (X['years_since_founding'] + 1)
X['funding_per_year'] = X['funding_rounds_led'] / (X['years_with_startup'] + 1)
X['experience_ratio'] = X['years_with_startup'] / (X['founder_age'] + 1)
X['is_senior'] = (X['founder_age'] >= X['founder_age'].median())
X['high_revenue'] = (X['monthly_revenue_generated'] >= X['monthly_revenue_generated'].median())
X['is_experienced'] = (X['years_with_startup'] >= X['years_with_startup'].median())
X['has_funding'] = (X['funding_rounds_led'] > 0)
X['far_from_hub'] = (X['distance_from_investor_hub'] >= X['distance_from_investor_hub'].median())
X['old_startup'] = (X['years_since_founding'] >= X['years_since_founding'].median())
X['log_revenue'] = np.log1p(X['monthly_revenue_generated'])
X['log_funding'] = np.log1p(X['funding_rounds_led'])

X_test['age_x_experience'] = X_test['founder_age'] * X_test['years_with_startup']
X_test['revenue_per_year'] = X_test['monthly_revenue_generated'] / (X_test['years_since_founding'] + 1)
X_test['funding_per_year'] = X_test['funding_rounds_led'] / (X_test['years_with_startup'] + 1)
X_test['experience_ratio'] = X_test['years_with_startup'] / (X_test['founder_age'] + 1)
X_test['is_senior'] = (X_test['founder_age'] >= X['founder_age'].median())
X_test['high_revenue'] = (X_test['monthly_revenue_generated'] >= X['monthly_revenue_generated'].median())
X_test['is_experienced'] = (X_test['years_with_startup'] >= X['years_with_startup'].median())
X_test['has_funding'] = (X_test['funding_rounds_led'] > 0)
X_test['far_from_hub'] = (X_test['distance_from_investor_hub'] >= X['distance_from_investor_hub'].median())
X_test['old_startup'] = (X_test['years_since_founding'] >= X['years_since_founding'].median())
X_test['log_revenue'] = np.log1p(X_test['monthly_revenue_generated'])
X_test['log_funding'] = np.log1p(X_test['funding_rounds_led'])

numerical_cols = [
    'founder_age',
    'years_with_startup',
    'monthly_revenue_generated',
    'funding_rounds_led',
    'distance_from_investor_hub',
    'num_dependents',
    'years_since_founding',
]

added_numeric_cols = [
    'age_x_experience',
    'revenue_per_year',
    'funding_per_year',
    'experience_ratio',
    'log_revenue',
    'log_funding'
]

# Remove Outliers

for cols in [numerical_cols,added_numeric_cols]:
  for col in cols:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Clip both Train and Test to the Train boundaries
    X[col] = X[col].clip(lower=lower_bound, upper=upper_bound)
    X_test[col] = X_test[col].clip(lower=lower_bound, upper=upper_bound)

# Scale the numeric data

scaler = PowerTransformer(method='yeo-johnson')
X[numerical_cols+added_numeric_cols] = scaler.fit_transform(X[numerical_cols+added_numeric_cols])
X_test[numerical_cols+added_numeric_cols] = scaler.transform(X_test[numerical_cols+added_numeric_cols])

In [2]:
""" TRAIN ONLY WITH WHOLE DATA AND CHECK VALIDATION,TEST SCORE """

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.8, random_state=42)

svm = SVC(
    kernel='poly',
    degree=2,
    C=4,
    probability=True,
    random_state=42,
)

# Train and predict (Validation)
svm.fit(X_train, y_train)
svm_y_pred = svm.predict(X_val)
svm_score = f1_score(y_val, svm_y_pred, average='weighted')
print("SVM F1 Score:", svm_score)

# Confusion Matrix for SVM
svm_cm = confusion_matrix(y_val, svm_y_pred)
plt.figure(figsize=(10, 8))
svm_disp = ConfusionMatrixDisplay(confusion_matrix=svm_cm,
                               display_labels=label_encoder.classes_)
svm_disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - SVM (Validation Set)')
plt.tight_layout()
plt.savefig('svm_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Predict test values
svm_test_pred = svm.predict(X_test)
svm_output_df = pd.DataFrame({'founder_id': test_ids, 'retention_status':label_encoder.inverse_transform(svm_test_pred)})

svm_output_df.to_csv('svm_submission20.csv', index=False)

nn = MLPClassifier(
    hidden_layer_sizes=(200, 100, 50),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size=256,
    learning_rate='adaptive',
    max_iter=300,
    early_stopping=True,
    random_state=42
)

nn.fit(X_train, y_train)
nn_y_pred = nn.predict(X_val)
nn_score = f1_score(y_val, nn_y_pred, average='weighted')
print("Neural Network F1 Score:", nn_score)

# Confusion matrix for NN
nn_cm = confusion_matrix(y_val, nn_y_pred)
plt.figure(figsize=(10, 8))
nn_disp = ConfusionMatrixDisplay(confusion_matrix=nn_cm,
                               display_labels=label_encoder.classes_)
nn_disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Neural Network (Validation Set)')
plt.tight_layout()
plt.savefig('nn_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Predict test values
nn_test_pred = nn.predict(X_test)
nn_output_df = pd.DataFrame({'founder_id': test_ids, 'retention_status': label_encoder.inverse_transform(nn_test_pred)})
nn_output_df.to_csv('nn_submission20.csv', index=False)

In [1]:
""" TRAIN ONLY WITH WHOLE DATA AND CHECK VALIDATION,TEST SCORE """

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LogisticRegression(random_state=42)

lr.fit(X_train, y_train)
lr_y_pred = lr.predict(X_val)
lr_score = f1_score(y_val, lr_y_pred, average='weighted')
print("Logistic Regression F1 Score:", lr_score)

# Confusion Matrix for SVM
lr_cm = confusion_matrix(y_val, lr_y_pred)
plt.figure(figsize=(10, 8))
svm_disp = ConfusionMatrixDisplay(confusion_matrix=lr_cm,
                               display_labels=label_encoder.classes_)
svm_disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Logistic Regression (Validation Set)')
plt.tight_layout()
plt.savefig('lr_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Predict test values
lr_test_pred = svm.predict(X_test)
lr_output_df = pd.DataFrame({'founder_id': test_ids, 'retention_status':label_encoder.inverse_transform(svm_test_pred)})

lr_output_df.to_csv('lr_submission.csv', index=False)

svm = SVC(
    kernel='poly',
    degree=2,
    C=4,
    probability=True,
    random_state=42,
)

# Train and predict (Validation)
svm.fit(X_train, y_train)
svm_y_pred = svm.predict(X_val)
svm_score = f1_score(y_val, svm_y_pred, average='weighted')
print("SVM F1 Score:", svm_score)

# Confusion Matrix for SVM
svm_cm = confusion_matrix(y_val, svm_y_pred)
plt.figure(figsize=(10, 8))
svm_disp = ConfusionMatrixDisplay(confusion_matrix=svm_cm,
                               display_labels=label_encoder.classes_)
svm_disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - SVM (Validation Set)')
plt.tight_layout()
plt.savefig('svm_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Predict test values
svm_test_pred = svm.predict(X_test)
svm_output_df = pd.DataFrame({'founder_id': test_ids, 'retention_status':label_encoder.inverse_transform(svm_test_pred)})

svm_output_df.to_csv('svm_submission.csv', index=False)

nn = MLPClassifier(
    hidden_layer_sizes=(200, 100, 50),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size=256,
    learning_rate='adaptive',
    max_iter=300,
    early_stopping=True,
    random_state=42
)

nn.fit(X_train, y_train)
nn_y_pred = nn.predict(X_val)
nn_score = f1_score(y_val, nn_y_pred, average='weighted')
print("Neural Network F1 Score:", nn_score)

# Confusion matrix for NN
nn_cm = confusion_matrix(y_val, nn_y_pred)
plt.figure(figsize=(10, 8))
nn_disp = ConfusionMatrixDisplay(confusion_matrix=nn_cm,
                               display_labels=label_encoder.classes_)
nn_disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Neural Network (Validation Set)')
plt.tight_layout()
plt.savefig('nn_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Predict test values
nn_test_pred = nn.predict(X_test)
nn_output_df = pd.DataFrame({'founder_id': test_ids, 'retention_status': label_encoder.inverse_transform(nn_test_pred)})
nn_output_df.to_csv('nn_submission.csv', index=False)

In [3]:
""" HYPERPARAMETER FINETUNING """

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
import pandas as pd

# Define the parameter grid for MLPClassifier
param_grid = {
    'hidden_layer_sizes': [(100,), (200, 100), (200, 100, 50)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001],
    'learning_rate': ['adaptive'],
    'max_iter': [300, 500]
}

# Initialize MLPClassifier with early stopping and random state
mlp = MLPClassifier(early_stopping=True, random_state=42, batch_size=256)

# Set up StratifiedKFold for cross-validation
# StratifiedKFold ensures that each fold has the same proportion of target classes as the complete set.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=skf,
    n_jobs=-1,  # Use all available CPU cores
    verbose=2
)

# Fit GridSearchCV to the training data (using the entire training set for grid search)
print("Starting GridSearchCV...")
grid_search.fit(X, y)

print("GridSearchCV completed.")

# Print the best parameters and best score
print("Best parameters found: ", grid_search.best_params_)
print("Best F1 Score found: ", grid_search.best_score_)

# Get the best model
best_mlp_model = grid_search.best_estimator_

# Predict on the test set using the best model
best_mlp_test_pred = best_mlp_model.predict(X_test)

# Create submission DataFrame
best_mlp_output_df = pd.DataFrame({'founder_id': test_ids, 'retention_status': label_encoder.inverse_transform(best_mlp_test_pred)})

# Save submission to CSV
best_mlp_output_df.to_csv('best_mlp_submission.csv', index=False)

print("Predictions with best MLP model saved to 'best_mlp_submission.csv'.")